In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

# <font color=blue>Stitching de Imágenes</font>


En este notebook aprenderás los conceptos y herramientas necesarias para unir dos imágenes y formar una panorámica.

## <font color="#F20C60">¿Qué es el stitching?</font>

Es el proceso de unir varias imágenes con zonas en común para formar una única imagen panorámica.

**Aplicaciones:**
- Fotografía panorámica.
- Mapas generados por drones.
- Realidad aumentada.

**Recomendaciones:**
- Asegúrate de que las imágenes tengan suficiente traslape.
- Usa imágenes tomadas desde el mismo punto de vista.
- ORB es más eficiente, pero menos preciso que SIFT.

## <font color="#F20C60"> Estimación de Homografía</font>

La **homografía** es una transformación que permite alinear las imágenes en el mismo plano.

### <font color="#770EB2">Pasos</font>

1. Extraer puntos clave de los matches.
2. Usar `cv2.findHomography()` con RANSAC.

### <font color=green> Ejemplo </font>


Extraemos los puntos clave que se emparejaron

In [ ]:
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches[:50]]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches[:50]]).reshape(-1, 1, 2)

* `matches[:50]`: matches es una lista de coincidencias entre características detectadas en dos imágenes, generalmente obtenida con un algoritmo como ORB, SIFT o SURF. El numro representa la cantidad de coincidencias que se tomaran.

* `kp1[m.queryIdx].pt`: obtiene las coordenadas (x, y) del punto clave correspondiente en la imagen 1.
Se crea una lista de 50 puntos clave de la forma [(x1, y1), (x2, y2), ..., (x50, y50)].

* `[kp1[m.queryIdx].pt for m in matches[:50]]`: kp1 es la lista de `keypoints` (puntos clave) detectados en la imagen1. Cada coincidencia `m` en `matches` tiene un índice `queryIdx`, que indica la posición del punto clave en `kp1`.

* `np.float32([...])`: Convierte la lista de puntos clave en un array de NumPy con tipo de dato `float32`, necesario para `cv2.findHomography()`.

* `.reshape(-1, 1, 2)`: cambia la forma del array para que sea compatible.
  * -1 permite que NumPy determine automáticamente el número de filas.
  * 1 indica que cada punto debe estar en una estructura de lista anidada.
  * 2 representa las dos coordenadas (x, y) de cada punto.

Calcular homografía con RANSAC

In [ ]:
H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

Se asume que cada `src_pts[i]` corresponde a `dst_pts[i]`.

* **Método de estimación:**
  * `cv2.RANSAC` (RANdom SAmple Consensus): Método robusto contra outliers (puntos de correspondencia incorrectos). Usa una estrategia iterativa que selecciona subconjuntos aleatorios de puntos y encuentra la mejor transformación.
  * `cv2.LMEDS` (Least Median of Squares): Minimiza la mediana de los errores cuadrados en lugar del error medio cuadrático. Similar a RANSAC, pero sin necesidad de un umbral de reproyección.
  * `cv2.RHO` (M-Estimator Sample Consensus): Variante optimizada de RANSAC, más rápida en algunos casos. Utiliza una estrategia diferente para la selección de muestras, lo que reduce el tiempo de cómputo.
  * `0` (Método Directo) Método algebraico basado en la descomposición LU. No maneja outliers, simplemente encuentra la solución exacta para los puntos dados.
* **Umbral de reproyección:** Determina qué tan lejos puede estar un punto transformado del destino para considerarse válido. Cuanto menor sea este valor, más estricta será la validación, en este caso se usa `5.0`.


## <font color="#F20C60">Alineación de Imágenes</font>


Una vez obtenida la matriz de homografía, se puede transformar una imagen para alinear con la otra.

### <font color=green> Sintaxis de la función </font>
``` python
cv2.warpPerspective(src, H, dsize)
```

* `src:` Primera imagen.
* `H:` valor optenido en la homografia puede ser directa o con las transformaciones necesarias.
* `dsize:` medidas de la imagen en x y y.

### <font color=orange> Ejemplo </font>

Realizamos la alineación de la imagen con el metdodo `cv2.warpPerspective()` y superponemos img2 sobre result en la posición correcta.

In [ ]:
result = cv2.warpPerspective(img1, H, (img1.shape[1] + img2.shape[1], img1.shape[0]))
result[0:img2.shape[0], 0:img2.shape[1]] = img2

**Nota:** los pasos siguientes pueden varias de proceso y sol se aplican si al realizar `cv2.warpPerspective()`, genear un resultado de solo una imagen o secciones toda negra sin la imagen.

Obtiene la altura (h) y el ancho (w) de las imágenes img1 y img2.

In [ ]:
h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]

Definimos las coordenadas de los cuatro vértices de img1.

In [ ]:
corners_img1 = np.float32([[0, 0], [w1, 0], [0, h1], [w1, h1]]).reshape(-1, 1, 2)

Es necesario se plica la transformación de perspectiva con la estimación homografica a las esquinas de img1.

`transformed_corners` contiene las nuevas posiciones de los vértices en img2.

In [ ]:
transformed_corners = cv2.perspectiveTransform(corners_img1, H12)

Encontrar los límites de la nueva imagen transformada:
 * `min_x y min_y:` las coordenadas mínimas (se comparan con 0 para evitar valores negativos).
 * `max_x y max_y:` las coordenadas máximas (se comparan con `w2` y `h2` para incluir img2).

In [ ]:
min_x = min(transformed_corners[:, 0, 0].min(), 0)
min_y = min(transformed_corners[:, 0, 1].min(), 0)
max_x = max(transformed_corners[:, 0, 0].max(), w2)
max_y = max(transformed_corners[:, 0, 1].max(), h2)

Calculamos el nuevo tamaño de la imagen resultante después de la transformación.

In [ ]:
new_w = int(max_x - min_x)
new_h = int(max_y - min_y)

Ajustamos la imagen para que min_x y min_y sean 0.

In [ ]:
translation_matrix = np.array([[1, 0, -min_x], [0, 1, -min_y], [0, 0, 1]])

Corrigimos la homografía H para evitar desplazamientos negativos.

In [ ]:
H_adj = np.dot(translation_matrix, H)

Y ahora si ya se puede aplicar `cv2.warpPerspective()` como se vio anteriormente, con los valores transformados.

## <font color="#F20C60">Metodo Stitcher</font>

Este metodo realiza todos los pasos necesarios para realizar una union entre imagenes de manera correcta sin necesidad de nosotros tener que generar las caracteristicas y maching de imagenes.

### <font color=green> Sintaxis de la función </font>

``` python
stitchy=cv2.Stitcher.create()
(dummy,output)=stitchy.stitch(imgs)
```

Se crea el stitcher de igual manera en como se crean los metodos de detección de esquinas.

Por ultimo ejecutamos `stitch()`, que intenta unir las imágenes:
* `dummy`: Estado del proceso (indica éxito o error).
* `output`: Imagen final combinada.
* `imgs`: Lista de imagenes a unir.
